In [1]:
import sys
import os
# Agrega el directorio padre (Source) al sys.path
sys.path.append(os.path.abspath('..'))
from raidcontrol.NumberCNN_OCR import DigitCNN
import os
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
@dataclass
class Config:
    ckpt_path: str = "digit_cnn_64_v4.pt"
    test_dir: str = r"E:\Documents\Facultad\Proyecto\Data\dataset\synthetic_number_test\images"  # 
    batch_size: int = 256
    num_workers: int = 0  # Windows safe
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    topk_errors: int = 30

In [3]:
def compute_confusion_matrix(y_true, y_pred, num_classes=10):
    cm = torch.zeros((num_classes, num_classes), dtype=torch.int64)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm


def precision_recall_f1_from_cm(cm: torch.Tensor):
    # cm: [C,C] where rows=true, cols=pred
    C = cm.size(0)
    tp = torch.diag(cm).float()
    fp = cm.sum(dim=0).float() - tp
    fn = cm.sum(dim=1).float() - tp

    precision = tp / (tp + fp + 1e-12)
    recall = tp / (tp + fn + 1e-12)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)

    support = cm.sum(dim=1).float()
    macro_f1 = f1.mean().item()
    weighted_f1 = (f1 * (support / (support.sum() + 1e-12))).sum().item()

    return precision, recall, f1, macro_f1, weighted_f1


def main():
    cfg = Config()

    # Load checkpoint
    ckpt = torch.load(cfg.ckpt_path, map_location="cpu")
    model = DigitCNN()
    model.load_state_dict(ckpt["model_state"])
    model.to(cfg.device)
    model.eval()

    # Test transforms (no augmentation)
    test_tfms = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(),
    ])

    test_ds = datasets.ImageFolder(cfg.test_dir, transform=test_tfms)
    test_loader = DataLoader(
        test_ds,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=True,
    )

    criterion = nn.CrossEntropyLoss(reduction="sum")  # sum for average later

    all_true = []
    all_pred = []
    all_conf = []
    all_paths = []

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(cfg.device, non_blocking=True)
            y = y.to(cfg.device, non_blocking=True)

            logits = model(x)
            loss = criterion(logits, y)
            total_loss += loss.item()

            probs = torch.softmax(logits, dim=1)
            conf, pred = torch.max(probs, dim=1)

            total_correct += (pred == y).sum().item()
            total_samples += y.size(0)

            all_true.append(y.cpu())
            all_pred.append(pred.cpu())
            all_conf.append(conf.cpu())

    all_true = torch.cat(all_true)
    all_pred = torch.cat(all_pred)
    all_conf = torch.cat(all_conf)

    avg_loss = total_loss / max(1, total_samples)
    acc = total_correct / max(1, total_samples)

    cm = compute_confusion_matrix(all_true.tolist(), all_pred.tolist(), num_classes=10)
    precision, recall, f1, macro_f1, weighted_f1 = precision_recall_f1_from_cm(cm)

    print("\n=== Test Metrics ===")
    print(f"Samples: {total_samples}")
    print(f"Loss: {avg_loss:.6f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Macro F1: {macro_f1:.4f}")
    print(f"Weighted F1: {weighted_f1:.4f}")

    print("\nPer-class (digit) metrics:")
    for d in range(10):
        print(f"{d}: P={precision[d].item():.3f} R={recall[d].item():.3f} F1={f1[d].item():.3f} "
              f"(support={cm[d].sum().item()})")

    print("\nConfusion matrix (rows=true, cols=pred):")
    print(cm)

    # Show top confident mistakes (very useful)
    mistakes = (all_pred != all_true)
    if mistakes.any():
        idxs = torch.where(mistakes)[0]
        # Sort mistakes by confidence descending
        sorted_idxs = idxs[torch.argsort(all_conf[idxs], descending=True)]
        k = min(cfg.topk_errors, sorted_idxs.numel())

        print(f"\nTop {k} most confident mistakes:")
        for i in range(k):
            idx = sorted_idxs[i].item()
            t = int(all_true[idx].item())
            p = int(all_pred[idx].item())
            c = float(all_conf[idx].item())
            # Path mapping: ImageFolder keeps list of (path, class_idx) in .samples
            path = test_ds.samples[idx][0]
            print(f"- true={t} pred={p} conf={c:.3f} | {path}")
    else:
        print("\nNo mistakes found. Nice!")

In [4]:
main()

e:\Documents\Facultad\Proyecto\raidcontrol\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



=== Test Metrics ===
Samples: 5000
Loss: 0.149542
Accuracy: 0.9640
Macro F1: 0.9643
Weighted F1: 0.9642

Per-class (digit) metrics:
0: P=0.979 R=0.970 F1=0.975 (support=474)
1: P=0.992 R=0.971 F1=0.981 (support=486)
2: P=0.889 R=0.998 F1=0.940 (support=520)
3: P=0.996 R=0.957 F1=0.976 (support=517)
4: P=0.975 R=0.973 F1=0.974 (support=489)
5: P=0.941 R=0.939 F1=0.940 (support=523)
6: P=0.966 R=0.984 F1=0.975 (support=512)
7: P=0.979 R=0.967 F1=0.973 (support=519)
8: P=0.974 R=0.941 F1=0.957 (support=475)
9: P=0.968 R=0.936 F1=0.952 (support=485)

Confusion matrix (rows=true, cols=pred):
tensor([[460,   0,   0,   0,   0,   2,   6,   0,   0,   6],
        [  2, 472,   3,   0,   2,   1,   0,   5,   0,   1],
        [  0,   0, 519,   0,   0,   0,   0,   1,   0,   0],
        [  0,   0,  18, 495,   0,   3,   1,   0,   0,   0],
        [  1,   0,   5,   0, 476,   4,   2,   1,   0,   0],
        [  0,   0,  24,   1,   1, 491,   2,   0,   2,   2],
        [  4,   0,   0,   0,   0,   3, 504,  